# Method 1 — Part 1: redundancy between TESSERA and AlphaEarth

**Question:** before fusing them, how much do the two embeddings share? If AlphaEarth were almost a linear function of TESSERA (or the reverse), concatenating them could hardly help a linear head; the room for fusion is what one embedding holds that the other cannot express.

**Measures** (about 100,000 valid pixels from the training patches of fold I, the same pixels for every embedding):
- **linear CKA**: 1 = same geometry up to rotation and scale, 0 = unrelated. Reference: *Better Together* (2026) reports CKA 0.42–0.60 between every pair of Earth embeddings, at point level;
- **canonical correlations (CCA)**: how many directions the two spaces share (values near 1) and how many are specific (values near 0);
- **held-out linear R²** in both directions: share of the variance of one embedding predicted by a ridge regression on the other, on patches not used for fitting;
- **intrinsic dimension**: number of principal components for 90% / 95% of the variance, for each embedding and for the concatenation TESSERA 2019 + AlphaEarth 2019.

Pairs: T 2019 / AE 2019 (the fusion of Method 1), T 2018 / AE 2018, and each embedding between 2018 and 2019 as a reference. Each measure is given on all valid pixels and on crop pixels only (labels 1–18).

**Needs:** Parts 1–7 (in particular `processed/valid_mask.npy`). **Runtime:** CPU (High-RAM not needed). **Time:** about 5–10 min, mostly reading the sampled patches from the Drive.

## 1. Runtime, Drive and configuration

In [ ]:
import sys, platform
from pathlib import Path
import json, pandas as pd
print("Python", platform.python_version())

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
OUT = DRIVE / "results" / "m1"

## 2. Code and tests
**Check:** all tests pass (`14 passed` or more).

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests

## 3. Redundancy measures
**Check:** two tables are printed and `results/m1/` holds `m1_redundancy_pairs.csv`, `m1_redundancy_dims.csv`, `m1_redundancy.png` and `m1_redundancy_info.json`.

In [ ]:
!python scripts/m1_redundancy.py --processed "{PROC_DRIVE}" --out "{OUT}"

In [ ]:
from IPython.display import Image, display
display(Image(str(OUT / "m1_redundancy.png")))

## 4. How to read the results
- **CKA of T 2019 / AE 2019** compared with 0.42–0.60 (*Better Together*, point level) and with the CKA of each embedding between 2018 and 2019.
- **CCA:** few canonical correlations above 0.9 and many below 0.5 mean that each embedding has its own directions, which a fusion can exploit.
- **R² in both directions:** if R² (AE from T) is high (e.g. > 0.8) but R² (T from AE) is low, AlphaEarth is largely contained in TESSERA and the gain of the fusion would have to come from the rest; similar moderate values in both directions mean real complementarity.
- **Crop pixels vs all pixels:** the task is crop mapping; redundancy measured mostly on background could be misleading.
- **Intrinsic dimension:** if the concatenation needs many fewer components than 128 + 64 for 95% of the variance, a bottleneck or PCA fusion loses little variance.